# Clustering of TOP 5 European Leagues Players - 2022/23
**Authors:** Cezary Kuźmowicz, Filip Żebrowski, Dariusz Doktorski

This notebook extends the 2021/22 reproduction analysis to the 2022/23 season. We proceed the same way as before: use the same feature set, preprocessing rules, clustering method, and evaluation plots, but load the newer season data.

[dataset used for analysis](https://www.kaggle.com/datasets/vivovinco/20222023-football-player-stats)

In [ ]:
from IPython.display import display

from football_clustering import (
    AnalysisConfig,
    add_cluster_labels,
    add_pca_components,
    cluster_feature_means,
    configure_notebook,
    evaluate_clusters,
    filter_by_minutes,
    fit_kmeans,
    hopkins_statistic,
    load_player_stats,
    plot_cluster_evaluation,
    plot_correlation_matrix,
    plot_distributions,
    plot_pca_clusters,
    plot_silhouette_profile,
    prepare_visualization_data,
    scale_features,
    select_features,
    validate_no_missing_values,
)

## Setup
Only the input file changes here. The shared configuration defaults and reusable analysis functions are defined in `football_clustering.py`.

In [ ]:
FILE_PATH = "data/2022-2023 Football Player Stats.csv"

config = AnalysisConfig(file_path=FILE_PATH)

SELECTED_FEATURES = list(config.selected_features)
VISUALIZATION_FEATURES = list(config.visualization_features)

configure_notebook(config.random_seed)

## Data Preparation
We load the 2022/23 data and apply the same minimum-minutes filter used in the first analysis.

In [ ]:
# Load the dataset using shared CSV formatting parameters.
raw_stats = load_player_stats(config)

In [ ]:
# Filter players based on the configured minimum minutes played.
stats_filtered = filter_by_minutes(raw_stats, config.min_minutes_played)

In [ ]:
display(stats_filtered.head())
display(stats_filtered.describe())
print(stats_filtered.info())

We also keep the same missing value validation step before clustering.

In [ ]:
# Print variables with missing values (if any) and fail before clustering if any exist.
missing_vals = validate_no_missing_values(stats_filtered)
print(missing_vals)
print("Data validation passed: No missing values found.")

## Feature Selection
The same eight clustering variables selected in the 2021/22 notebook are reused here, so the seasons can be compared under the same assumptions.

In [ ]:
reduced_stats = select_features(stats_filtered, SELECTED_FEATURES)
reduced_stats.head()

## Clusterability Check
As before, Hopkins statistic is used as a quick check that the selected feature space has clustering tendency.

In [ ]:
hopkins_val = hopkins_statistic(
    reduced_stats,
    sample_ratio=config.hopkins_sample_ratio,
    n_neighbors=config.hopkins_n_neighbors,
)
print(f"Hopkins Statistic: {hopkins_val:.4f}")

## Quick Visual Checks
The following plots repeat the same exploratory checks from the original notebook. The source CSVs use different `Goals` conventions: 2021/22 stores goals per 90, while 2022/23 stores total goals. The shared helper normalizes this before plotting `Distribution of Goals`, so this chart is comparable across seasons.

In [ ]:
player_stats = prepare_visualization_data(stats_filtered, VISUALIZATION_FEATURES)

In [ ]:
plot_distributions(player_stats)

The correlation matrix is retained to verify that the reused feature set is still reasonable for the newer season.

In [ ]:
plot_correlation_matrix(reduced_stats, "Correlation Matrix of Selected Features")

## Clustering
We standardize the selected features and compare candidate numbers of clusters. The silhouette check is used to identify the best standalone `k` for 2022/23, while the final model below still uses `k = 3` to keep the cluster interpretation comparable with the 2021/22 reproduction.

In [ ]:
stats_scaled, scaler = scale_features(reduced_stats, SELECTED_FEATURES)

display(stats_scaled.head())
display(stats_scaled.describe().round(2))

In [ ]:
cluster_evaluation = evaluate_clusters(
    stats_scaled,
    max_k=config.max_clusters,
    random_seed=config.random_seed,
    n_init=config.kmeans_n_init,
)

best_silhouette_k = int(
    cluster_evaluation.loc[
        cluster_evaluation["silhouette_score"].idxmax(),
        "k",
    ]
)

print(f"Best k by silhouette for 2022/23: {best_silhouette_k}")
display(cluster_evaluation.round(4))
plot_cluster_evaluation(cluster_evaluation)

### Standalone `k = 2` Clustering
The silhouette table selects `k = 2` as the strongest standalone split for the 2022/23 data. This model is useful for understanding the season on its own, while the later `k = 3` model is retained for direct comparison with the 2021/22 reproduction.

In [ ]:
kmeans_k2 = fit_kmeans(
    stats_scaled,
    n_clusters=best_silhouette_k,
    random_seed=config.random_seed,
    n_init=config.kmeans_n_init,
)

reduced_stats_k2 = add_cluster_labels(reduced_stats, kmeans_k2.labels_)

print("Cluster sizes for k = 2:")
display(reduced_stats_k2["Cluster"].value_counts().sort_index().to_frame("players"))